In [1]:
import requests
import json
import numpy as np
import pandas as pd
import pytz
from datetime import date, datetime, timedelta
import concurrent.futures
import base64
import os
import time
import duckdb

# Suppress SettingWithCopyWarning
pd.options.mode.chained_assignment = None
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)

In [ ]:
game_summary = pd.read_csv(r"C:\Users\kjchr\Desktop\hockey\new_hockey_env\API data\Base Data\Base Data\Game Summary\all_game_summary.csv")
game_summary = game_summary.rename(columns={'season':'season_id'})
game_summary.head()

In [2]:
con = duckdb.connect("pbp.duckdb")

# con.execute(r"""
#     CREATE TABLE game_plays AS
#     SELECT * FROM read_csv_auto(
#         'C:\\Users\\kjchr\\Desktop\\hockey\\new_hockey_env\\API data\\Base Data\\Play By Play\\history_play_by_play.csv'
#     );
# """)

pbp = con.execute("SELECT * FROM game_plays").df()
pbp = pbp.rename(columns={'event_team_id':'team_id'})


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [3]:
pbp[["game_id", "eventId"]].drop_duplicates().shape


(3915161, 2)

In [28]:
pbp[pbp['game_id']==2013020001].head()

,season_id,game_id,eventId,timeInPeriod,timeRemaining,situationCode,homeTeamDefendingSide,typeCode,typeDescKey,sortOrder,period_number,period_type,max_regulation_periods,team_id,details.losingPlayerId,winning_player_id,x_coord,y_coord,zone_code,hitter,hittee,shot_type,shooting_player,goalie,away_sog,home_sog,player_id,reason,blocking_player,type_code,desc_key,duration,committed_by,drawn_by,secondary_reason,pptReplayUrl,scoring_player,scoring_player_total,assist_1,assist1_total,away_score,home_score,details.highlightClipSharingUrl,details.highlightClip,details.discreteClip,assist_2,assist2_total,served_by,details.discreteClipFr,details.highlightClipSharingUrlFr,details.highlightClipFr,situation,goalie_situation,game_in_seconds
0,20132014,2013020001,51,00:00:00,20:00:00,NaN,None,520,period-start,5,1,REG,3,NaN,NaN,NaN,NaN,NaN,None,NaN,NaN,None,NaN,NaN,NaN,NaN,NaN,None,NaN,None,None,NaN,NaN,NaN,None,None,NaN,NaN,NaN,NaN,NaN,NaN,None,None,None,NaN,NaN,NaN,None,None,None,None,in net,0
1,20132014,2013020001,280,00:00:00,20:00:00,1551.0,None,502,faceoff,12,1,REG,3,10.0,8469521.0,8475098.0,0.0,0.0,N,NaN,NaN,None,NaN,NaN,NaN,NaN,NaN,None,NaN,None,None,NaN,NaN,NaN,None,None,NaN,NaN,NaN,NaN,NaN,NaN,None,None,None,NaN,NaN,NaN,None,None,None,5 on 5,in net,0
2,20132014,2013020001,53,01:14:00,18:46:00,1551.0,None,507,missed-shot,13,1,REG,3,10.0,NaN,NaN,35.0,-28.0,O,NaN,NaN,wrist,8471742.0,8471679.0,NaN,NaN,NaN,wide-of-net,NaN,None,None,NaN,NaN,NaN,None,None,NaN,NaN,NaN,NaN,NaN,NaN,None,None,None,NaN,NaN,NaN,None,None,None,5 on 5,in net,74
3,20132014,2013020001,54,01:36:00,18:24:00,1551.0,None,506,shot-on-goal,18,1,REG,3,8.0,NaN,NaN,-82.0,-22.0,O,NaN,NaN,wrist,8475848.0,8473503.0,0.0,1.0,NaN,None,NaN,None,None,NaN,NaN,NaN,None,None,NaN,NaN,NaN,NaN,NaN,NaN,None,None,None,NaN,NaN,NaN,None,None,None,5 on 5,in net,96
4,20132014,2013020001,5,01:40:00,18:20:00,1551.0,None,503,hit,19,1,REG,3,8.0,NaN,NaN,-97.0,20.0,O,8476851.0,8474581.0,None,NaN,NaN,NaN,NaN,NaN,None,NaN,None,None,NaN,NaN,NaN,None,None,NaN,NaN,NaN,NaN,NaN,NaN,None,None,None,NaN,NaN,NaN,None,None,None,5 on 5,in net,100


In [4]:
rush_pbp = con.execute("""
    SELECT 
        season_id,
        game_id,
        eventId,
        timeInPeriod,
        timeRemaining,
        typeCode,
        typeDescKey,
        sortOrder,
        period_number,
        event_team_id AS team_id,
        x_coord,
        y_coord,
        zone_code,
        game_in_seconds
    FROM game_plays
    WHERE typeDescKey IS NOT NULL
""").df()
rush_pbp[["game_id", "eventId"]].drop_duplicates().shape

(3915161, 2)

In [9]:
dfs = [game_summary, rush_pbp, pbp]
exclude_cols = ["x_coord", "y_coord"]

for i, df in enumerate(dfs):
    float_cols = df.select_dtypes(include='float').columns
    float_cols = float_cols.difference(exclude_cols)
    dfs[i][float_cols] = df[float_cols].astype("Int64")



KeyboardInterrupt



In [ ]:
# --------------------------------------
# 1. Define event groups
# --------------------------------------
shot_events = {
    "shot-on-goal", 
    "missed-shot", 
    "goal", 
    "blocked-shot", 
    "failed-shot-attempt"
}

transition_events = {
    "giveaway",
    "takeaway",
    "hit"
}

stoppage_events = {
    "faceoff",
    "stoppage",
    "period-end",
    "game-end",
    "penalty",
    "delayed-penalty",
}

valid_prev_events = shot_events | transition_events | stoppage_events

# --------------------------------------
# 2. Filter to only events we care about
# --------------------------------------
mask = rush_pbp["typeDescKey"].isin(valid_prev_events)
valid_plays = rush_pbp.loc[mask].copy()


season_id          303
game_id            303
eventId            303
timeInPeriod       303
timeRemaining      303
typeCode           303
typeDescKey        303
sortOrder          303
period_number      303
team_id            253
x_coord            253
y_coord            253
zone_code          253
game_in_seconds    303
dtype: int64

In [37]:

# Sort by game + time
valid_plays = valid_plays.sort_values(["game_id", "sortOrder"])

# --------------------------------------
# 3. Build "previous event" context
# --------------------------------------
grouped = valid_plays.groupby("game_id", sort=False)

valid_plays["prev_event_time"] = grouped["game_in_seconds"].shift(1)
valid_plays["prev_event_type"] = grouped["typeDescKey"].shift(1)
valid_plays["prev_zone"]       = grouped["zone_code"].shift(1)
valid_plays["prev_team_id"]    = grouped["team_id"].shift(1)

valid_plays["prev_event_was_stoppage"] = valid_plays["prev_event_type"].isin(stoppage_events)
valid_plays["time_diff"] = valid_plays["game_in_seconds"] - valid_plays["prev_event_time"]
# --------------------------------------
# 4. Rush shot condition
# --------------------------------------
is_shot = valid_plays["typeDescKey"].isin(shot_events)

# Time window: allow 0–5 seconds
quick = valid_plays["time_diff"].between(0, 5, inclusive="both")

# Possession change: previous team != shooting team
possession_change = (
    valid_plays["prev_team_id"].notna() &
    (valid_plays["prev_team_id"] != valid_plays["team_id"])
)

# Same team, but coming from DZ or NZ
same_team_dz_nz = (
    (valid_plays["prev_team_id"] == valid_plays["team_id"]) &
    (valid_plays["prev_zone"].isin(["D", "N"]))
)

no_stoppage = ~valid_plays["prev_event_was_stoppage"]

rush_mask = (
    is_shot &
    quick &
    no_stoppage &
    (possession_change | same_team_dz_nz)
)

rush_df = valid_plays.loc[rush_mask].copy()

# Deduplicate: some feeds log multiple rows per eventId
# rush_df = rush_df.drop_duplicates(subset=["game_id", "eventId"])

rush_df["rush_shot"] = 1


In [38]:
game_event_comp = rush_df[rush_df['game_id']==2013020100]
game_event_comp['game_id'].count()

np.int64(6)

In [34]:
# --------------------------------------
# 5. Rush shots FOR (team taking the rush shot)
# --------------------------------------
rush_shots_for = (
    rush_df
    .groupby(["season_id", "team_id"])
    .size()
    .reset_index(name="rush_shots_for")
)

# --------------------------------------
# 6. Rush shots AGAINST (team whose previous event enabled the rush)
# --------------------------------------
rush_shots_against = (
    rush_df
    .groupby(["season_id", "prev_team_id"])
    .size()
    .reset_index(name="rush_shots_against")
    .rename(columns={"prev_team_id": "team_id"})
)

# --------------------------------------
# 7. Merge FOR + AGAINST
# --------------------------------------
rush_stats = rush_shots_for.merge(
    rush_shots_against,
    on=["season_id", "team_id"],
    how="outer"
)

rush_stats = rush_stats.fillna(0)

rush_stats["rush_shots_for"] = rush_stats["rush_shots_for"].astype(int)
rush_stats["rush_shots_against"] = rush_stats["rush_shots_against"].astype(int)

rush_stats["rush_shot_differential"] = (
    rush_stats["rush_shots_for"] - rush_stats["rush_shots_against"]
)

rush_stats = (
    rush_stats
    .sort_values(["season_id", "rush_shot_differential"], ascending=[True, False])
    .reset_index(drop=True)
)

rush_stats.head()


,season_id,team_id,rush_shots_for,rush_shots_against,rush_shot_differential
0,20132014,16.0,511,395,116
1,20132014,28.0,529,458,71
2,20132014,1.0,357,298,59
3,20132014,23.0,458,409,49
4,20132014,17.0,417,372,45


In [35]:
rush_df["sortOrder"].nunique(), len(rush_df)


(995, 187748)

In [ ]:
# Check if PBP was duplicated BEFORE rush logic
rush_pbp['eventId'].value_counts().head()
rush_pbp['eventId'].duplicated().sum()
rush_pbp.shape

